# GPU kernels, a campaign: the notebook

This is the hands-on side of the campaign at **https://notes.ybc.sh/play/gpu/**. Read a level there first, then come here to write the kernel,
time it and compare it with cuBLAS on your own GPU.

**Colab:** Runtime > Change runtime type > **T4 GPU** (free), then run the cells top to bottom.
**Your own GPU:** Jupyter with `pip install cupy-cuda12x` (CUDA 12 drivers).

At the end, the last cell prints a **score line**: paste it into the ladder on the campaign pages.

## Setup: which GPU are we on?

In [ ]:
import subprocess, sys
try:
    import cupy as cp
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
    import cupy as cp
import numpy as np

props = cp.cuda.runtime.getDeviceProperties(0)
GPU = props["name"].decode() if isinstance(props["name"], bytes) else str(props["name"])
sms = props["multiProcessorCount"]
print(f"GPU: {GPU}   SMs: {sms}   compute capability {props['major']}.{props['minor']}")
mem_khz, bus_bits = props.get("memoryClockRate"), props.get("memoryBusWidth")
if mem_khz and bus_bits:
    print(f"theoretical memory bandwidth ~ {2 * mem_khz * 1e3 * bus_bits / 8 / 1e9:.0f} GB/s")

## The harness: data, a reference answer, and a stopwatch

In [ ]:
N = 4096                                   # C = A x B, all N x N float32
A = cp.random.standard_normal((N, N), dtype=cp.float32)
B = cp.random.standard_normal((N, N), dtype=cp.float32)
C_ref = A @ B                              # cuBLAS: the number to chase
FLOPS = 2 * N**3                           # one multiply + one add per term
SCORES = {}

def time_ms(launch, reps=3):
    launch(); cp.cuda.Device().synchronize()          # warm-up (and first-call compilation)
    start, end = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    for _ in range(reps):
        launch()
    end.record(); end.synchronize()
    return cp.cuda.get_elapsed_time(start, end) / reps

out = cp.empty_like(A)
cublas_ms = time_ms(lambda: cp.matmul(A, B, out=out))
SCORES["cublas"] = FLOPS / (cublas_ms * 1e-3) / 1e9
print(f"cuBLAS: {cublas_ms:.2f} ms = {SCORES['cublas']:.0f} GFLOP/s   <- 100% on your ladder")

def run(src, name, label, block=(32, 32)):
    kernel = cp.RawKernel(src, name)
    grid = ((N + block[0] - 1) // block[0], (N + block[1] - 1) // block[1])
    C = cp.zeros_like(A)
    ms = time_ms(lambda: kernel(grid, block, (A, B, C, cp.int32(N))))
    err = float(cp.linalg.norm(C - C_ref) / cp.linalg.norm(C_ref))
    gflops = FLOPS / (ms * 1e-3) / 1e9
    if err > 1e-4:
        print(f"{label}: WRONG ANSWER (relative error {err:.1e}); not scored. Check your indices.")
        return
    SCORES[label] = gflops
    print(f"{label}: {ms:.1f} ms = {gflops:.0f} GFLOP/s = {100 * gflops / SCORES['cublas']:.1f}% of cuBLAS   (correct, error {err:.1e})")

## Level 1: the 1% kernel ([story](https://notes.ybc.sh/play/gpu/level-1.html))

First the kernel as given: one thread per element of C. Before running it, write down your guess: what % of cuBLAS?

In [ ]:
naive_src = r"""
// Level 1, as given: one thread per output element of C = A x B (N x N, row-major).
extern "C" __global__ void matmul_naive(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;   // threadIdx.x walks DOWN a column of C
    int col = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < N && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < N; ++k)
            acc += A[row * N + k] * B[k * N + col];
        C[row * N + col] = acc;
    }
}
"""
run(naive_src, "matmul_naive", "naive")

**Your move.** The kernel below is the same one, renamed. Change it so the 32 threads of a warp read contiguous
memory. It is a two-line change. Predict the speedup first, then run.

In [ ]:
mine_src = r"""
// Level 1, your move: change which index follows threadIdx.x so a warp reads contiguous memory.
extern "C" __global__ void matmul_mine(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;   // threadIdx.x walks DOWN a column of C
    int col = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < N && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < N; ++k)
            acc += A[row * N + k] * B[k * N + col];
        C[row * N + col] = acc;
    }
}
"""
run(mine_src, "matmul_mine", "coalesced")

## Level 2: stop fetching the same numbers ([story](https://notes.ybc.sh/play/gpu/level-2.html))

Fill in the three TODOs: cooperative tile loads, then two barriers. The skeleton compiles but is wrong until you do.
Predict first: tiling cuts global-memory reads by 32x. How much faster than your coalesced kernel will it be?

In [ ]:
tiled_src = r"""
// Level 2, your move: fill in the three TODOs. The skeleton compiles as is but gives wrong results.
#define TILE 32
extern "C" __global__ void matmul_tiled_mine(const float* A, const float* B, float* C, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;
    int col = blockIdx.x * TILE + tx;
    float acc = 0.0f;
    for (int t = 0; t < N; t += TILE) {
        // TODO 1: each thread loads ONE element of the A tile and ONE of the B tile.
        //         Tile of A: rows of this block, columns t..t+TILE-1. Tile of B: rows t..t+TILE-1, columns of this block.
        //         Keep neighbouring threads (tx, tx+1) on neighbouring addresses. Write 0.0f when out of bounds.
        As[ty][tx] = 0.0f;
        Bs[ty][tx] = 0.0f;
        // TODO 2: a barrier, so the tile is complete before anyone reads it.
        for (int k = 0; k < TILE; ++k)
            acc += As[ty][k] * Bs[k][tx];
        // TODO 3: a second barrier. Why is it needed? (Level 2 explains.)
    }
    if (row < N && col < N)
        C[row * N + col] = acc;
}
"""
run(tiled_src, "matmul_tiled_mine", "tiled")

## Your score line: paste it into the ladder on the campaign pages

In [ ]:
line = ";".join([f"gpu={GPU}"] + [f"{k}={v:.1f}" for k, v in SCORES.items()])
print(line)

---
## Spoilers: the reference solutions

Run these only after trying. They also let you compare your numbers with a known-good kernel.

In [ ]:
ref_coalesced = r"""
// Level 1, solved: consecutive threads of a warp now own consecutive columns.
extern "C" __global__ void matmul_coalesced(const float* A, const float* B, float* C, int N) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;   // threadIdx.x walks ALONG a row of C
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < N && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < N; ++k)
            acc += A[row * N + k] * B[k * N + col];
        C[row * N + col] = acc;
    }
}
"""
run(ref_coalesced, "matmul_coalesced", "ref_coalesced")

In [ ]:
ref_tiled = r"""
// Level 2, solved: each block stages 32x32 tiles of A and B in shared memory.
#define TILE 32
extern "C" __global__ void matmul_tiled(const float* A, const float* B, float* C, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;
    int col = blockIdx.x * TILE + tx;
    float acc = 0.0f;
    for (int t = 0; t < N; t += TILE) {
        // Cooperative, coalesced loads: neighbouring threads read neighbouring addresses.
        As[ty][tx] = (row < N && t + tx < N) ? A[row * N + t + tx] : 0.0f;
        Bs[ty][tx] = (t + ty < N && col < N) ? B[(t + ty) * N + col] : 0.0f;
        __syncthreads();                    // the tile is complete before anyone reads it
        for (int k = 0; k < TILE; ++k)
            acc += As[ty][k] * Bs[k][tx];
        __syncthreads();                    // nobody overwrites the tile while others still read it
    }
    if (row < N && col < N)
        C[row * N + col] = acc;
}
"""
run(ref_tiled, "matmul_tiled", "ref_tiled")